# Supervised AO2D Model Inspection

This notebook checks the current supervised model checkpoint, plots training curves, and visualizes validation/test predictions.

Default target: `configs/supervised_rcan2d.json` -> `outputs/rcan2d_supervised_v2/best.pt`.

## 1. Setup

If a dependency import fails, install the project environment first, for example:

```bash
pip install -e . matplotlib
```

In [ ]:
from __future__ import annotations

import json
import math
import random
import sys
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "ao2d").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from ao2d.data import AO2DPairDataset, build_dataloader, get_data_root, resolve_path
from ao2d.models.factory import make_model
from ao2d.training.metrics import psnr, ssim

torch.set_grad_enabled(False)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True

print(f"repo: {REPO_ROOT}")
print(f"torch: {torch.__version__}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Select Config And Checkpoint

Change `CONFIG_PATH`, `OUTPUT_DIR`, `CHECKPOINT_NAME`, or `DATA_ROOT_OVERRIDE` here when you want to inspect another run.

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "supervised_rcan2d.json"
CHECKPOINT_NAME = "best.pt"  # best.pt or last.pt
OUTPUT_DIR = None  # None means: read config['output_dir']
DATA_ROOT_OVERRIDE = "/mnt/share/dyx/Data/Data2d"  # e.g. "/path/to/data"; None uses config/env/current paths
EVAL_SPLIT = "test"  # "test", "val", "train", or "auto"
ALLOW_EVAL_SPLIT_FALLBACK = True  # fall back to val/train while Test is not uploaded yet

with CONFIG_PATH.open("r") as f:
    config = json.load(f)

output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else REPO_ROOT / config.get("output_dir", "outputs/supervised")
checkpoint_path = output_dir / CHECKPOINT_NAME
metrics_path = output_dir / "metrics.xlsx"
data_root = get_data_root(config, DATA_ROOT_OVERRIDE)

print(f"config:     {CONFIG_PATH}")
print(f"output_dir: {output_dir}")
print(f"checkpoint: {checkpoint_path} ({'exists' if checkpoint_path.exists() else 'missing'})")
print(f"metrics:    {metrics_path} ({'exists' if metrics_path.exists() else 'missing'})")
available_data_splits = [k for k, v in config.get("data", {}).items() if isinstance(v, dict)]
print(f"data_root:  {data_root}")
print(f"splits:     {available_data_splits}")
print(f"eval split: {EVAL_SPLIT}")

## 3. Load Checkpoint And Inspect Parameters

In [ ]:
def tensor_stats(t: torch.Tensor) -> dict[str, float | tuple[int, ...]]:
    x = t.detach().float().cpu()
    return {
        "shape": tuple(t.shape),
        "mean": float(x.mean()) if x.numel() else math.nan,
        "std": float(x.std(unbiased=False)) if x.numel() else math.nan,
        "min": float(x.min()) if x.numel() else math.nan,
        "max": float(x.max()) if x.numel() else math.nan,
    }

def count_parameters(model: torch.nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

ckpt = torch.load(checkpoint_path, map_location="cpu")
ckpt_config = ckpt.get("config", config)
eval_config = json.loads(json.dumps(ckpt_config))
eval_config["data"] = config.get("data", ckpt_config.get("data", {}))
model = make_model(ckpt_config["model"])
load_result = model.load_state_dict(ckpt["model"], strict=False)
model.eval()

total_params, trainable_params = count_parameters(model)
print("checkpoint summary")
print(f"  epoch: {ckpt.get('epoch')}")
print(f"  val:   {ckpt.get('val')}")
print("model summary")
print(f"  name:       {ckpt_config['model'].get('name')}")
print(f"  params:     {total_params:,}")
print(f"  trainable:  {trainable_params:,}")
print(f"  missing keys:    {len(load_result.missing_keys)}")
print(f"  unexpected keys: {len(load_result.unexpected_keys)}")
if load_result.missing_keys:
    print("missing:", load_result.missing_keys[:20])
if load_result.unexpected_keys:
    print("unexpected:", load_result.unexpected_keys[:20])

largest = sorted(model.named_parameters(), key=lambda kv: kv[1].numel(), reverse=True)[:12]
print("\nlargest parameter tensors")
for name, param in largest:
    stats = tensor_stats(param)
    print(
        f"  {name:48s} {str(stats['shape']):18s} "
        f"n={param.numel():>9,} mean={stats['mean']:+.4g} std={stats['std']:.4g}"
    )

## 4. Plot Training Metrics

In [ ]:
def _xlsx_cell_value(cell: ET.Element, shared_strings: list[str]) -> str | float | int | None:
    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    if cell.attrib.get("t") == "inlineStr":
        text_node = cell.find(f"{ns}is/{ns}t")
        return text_node.text if text_node is not None else ""
    if cell.attrib.get("t") == "s":
        node = cell.find(f"{ns}v")
        return shared_strings[int(node.text)] if node is not None and node.text is not None else ""
    node = cell.find(f"{ns}v")
    if node is None or node.text is None:
        return None
    text = node.text
    try:
        value = float(text)
        return int(value) if value.is_integer() else value
    except ValueError:
        return text

def read_metrics_xlsx(path: Path) -> list[dict[str, object]]:
    if not path.exists():
        return []
    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    shared_strings: list[str] = []
    with zipfile.ZipFile(path) as zf:
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for item in root.findall(f"{ns}si"):
                texts = [node.text or "" for node in item.findall(f".//{ns}t")]
                shared_strings.append("".join(texts))
        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
    rows = []
    for row in root.findall(f".//{ns}row"):
        values = [_xlsx_cell_value(cell, shared_strings) for cell in row.findall(f"{ns}c")]
        rows.append(values)
    if not rows:
        return []
    headers = [str(x) for x in rows[0]]
    return [dict(zip(headers, row)) for row in rows[1:]]

metrics = read_metrics_xlsx(metrics_path)
print(f"loaded {len(metrics)} metric rows")
if metrics:
    print(metrics[-1])

In [ ]:
def column(rows: list[dict[str, object]], key: str) -> np.ndarray:
    values = []
    for row in rows:
        value = row.get(key)
        values.append(np.nan if value in (None, "") else float(value))
    return np.asarray(values, dtype=np.float32)

if not metrics:
    print("No metrics.xlsx found yet. Train the model first, then rerun this cell.")
else:
    epoch = column(metrics, "epoch")
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)

    axes[0, 0].plot(epoch, column(metrics, "train_loss"), label="train")
    axes[0, 0].plot(epoch, column(metrics, "val_loss"), label="val")
    axes[0, 0].set_title("L1 loss")
    axes[0, 0].set_xlabel("epoch")
    axes[0, 0].legend()

    axes[0, 1].plot(epoch, column(metrics, "train_psnr"), label="train")
    axes[0, 1].plot(epoch, column(metrics, "val_psnr"), label="val")
    axes[0, 1].set_title("PSNR")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].legend()

    axes[1, 0].plot(epoch, column(metrics, "train_ssim"), label="train")
    axes[1, 0].plot(epoch, column(metrics, "val_ssim"), label="val")
    axes[1, 0].set_title("SSIM")
    axes[1, 0].set_xlabel("epoch")
    axes[1, 0].legend()

    axes[1, 1].plot(epoch, column(metrics, "lr"), label="lr")
    axes[1, 1].set_title("Learning rate")
    axes[1, 1].set_xlabel("epoch")
    axes[1, 1].set_yscale("log")

    plt.show()

## 5. Build Validation/Test Loader

In [ ]:
def make_eval_dataset(config: dict, split: str = "val", data_root: Path | None = None):
    data_cfg = dict(config["data"][split])
    common = dict(
        patch_size=tuple(config["data"].get("patch_size", [256, 256])),
        augment=False,
        samples_per_epoch=data_cfg.get("samples_per_epoch"),
    )
    if "normalize_percentile" in config["data"]:
        common["normalize_percentile"] = tuple(config["data"]["normalize_percentile"])
    if "normalize_percentile" in data_cfg:
        common["normalize_percentile"] = tuple(data_cfg["normalize_percentile"])
    if "manifest" in data_cfg:
        return AO2DPairDataset.from_manifest(resolve_path(data_cfg["manifest"], data_root), data_root=data_root, **common)
    return AO2DPairDataset.from_dirs(
        resolve_path(data_cfg["aberrated_dir"], data_root),
        resolve_path(data_cfg["target_dir"], data_root),
        **common,
    )

def split_source_exists(config: dict, split: str, data_root: Path | None = None) -> tuple[bool, Path | None]:
    if split not in config.get("data", {}):
        return False, None
    data_cfg = config["data"][split]
    if "manifest" in data_cfg:
        path = resolve_path(data_cfg["manifest"], data_root)
        return path.exists(), path
    if "aberrated_dir" in data_cfg and "target_dir" in data_cfg:
        input_dir = resolve_path(data_cfg["aberrated_dir"], data_root)
        target_dir = resolve_path(data_cfg["target_dir"], data_root)
        return input_dir.exists() and target_dir.exists(), input_dir
    return True, None

def choose_eval_split(config: dict, requested: str, data_root: Path | None = None) -> str:
    candidates = ["test", "val", "train"] if requested == "auto" else [requested, "val", "train"]
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        exists, path = split_source_exists(config, candidate, data_root)
        if exists:
            if candidate != requested and requested != "auto":
                print(f"Requested split '{requested}' is unavailable at {path}; using '{candidate}' instead.")
            return candidate
        if candidate in config.get("data", {}):
            print(f"Split '{candidate}' is configured but not found yet: {path}")
        elif candidate == requested:
            print(f"Split '{candidate}' is not configured.")
        if requested != "auto" and not ALLOW_EVAL_SPLIT_FALLBACK:
            raise FileNotFoundError(f"Requested split '{requested}' is unavailable: {path}")
    raise ValueError("No usable data split found. Check DATA_ROOT_OVERRIDE and config['data'].")

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

split = choose_eval_split(eval_config, EVAL_SPLIT, data_root)
eval_set = make_eval_dataset(eval_config, split=split, data_root=data_root)
eval_loader = build_dataloader(eval_set, batch_size=1, shuffle=False, num_workers=0, drop_last=False)
print(f"split: {split}")
print(f"records: {len(eval_set.records)}")
print(f"samples_per_epoch: {len(eval_set)}")
print(f"first input:  {eval_set.records[0].aberrated}")
print(f"first target: {eval_set.records[0].target}")

## 6. Evaluate A Small Batch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

MAX_EVAL_SAMPLES = 16
rows = []
examples = []
for idx, batch in enumerate(eval_loader):
    if idx >= MAX_EVAL_SAMPLES:
        break
    x = batch["input"].to(device)
    y = batch["target"].to(device)
    out = model(x)
    pred = out[0] if isinstance(out, tuple) else out
    row = {
        "idx": idx,
        "l1": float(F.l1_loss(pred, y).cpu()),
        "psnr": float(psnr(y, pred).cpu()),
        "ssim": float(ssim(y, pred).cpu()),
        "pred_min": float(pred.amin().cpu()),
        "pred_max": float(pred.amax().cpu()),
        "input_path": batch["input_path"][0],
        "target_path": batch["target_path"][0],
    }
    rows.append(row)
    #not only the first 4 samples, but 4 samples evenly spaced across the eval set
    if idx%4 == 0 and len(examples) < 4:
        examples.append((x.cpu(), pred.cpu(), y.cpu(), row))

if rows:
    print(f"evaluated {len(rows)} samples")
    print(f"mean L1:   {np.mean([r['l1'] for r in rows]):.6f}")
    print(f"mean PSNR: {np.mean([r['psnr'] for r in rows]):.3f}")
    print(f"mean SSIM: {np.mean([r['ssim'] for r in rows]):.4f}")
    for row in rows[:5]:
        print(f"#{row['idx']:02d} L1={row['l1']:.5f} PSNR={row['psnr']:.2f} SSIM={row['ssim']:.4f} pred=[{row['pred_min']:.3f}, {row['pred_max']:.3f}]")
else:
    print("No evaluation rows were produced.")

## 7. Visualize Predictions

In [ ]:
def as_image(t: torch.Tensor) -> np.ndarray:
    return t.squeeze().detach().cpu().numpy()

if not examples:
    print("Run the evaluation cell first.")
else:
    n = len(examples)
    fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n), constrained_layout=True)
    if n == 1:
        axes = axes[None, :]
    for row_idx, (x, pred, y, meta) in enumerate(examples):
        images = [as_image(x), as_image(pred), as_image(y), np.abs(as_image(pred) - as_image(y))]
        titles = ["input", "prediction", "target", f"abs error\nL1={meta['l1']:.4f}"]
        for col_idx, (img, title) in enumerate(zip(images, titles)):
            ax = axes[row_idx, col_idx]
            im = ax.imshow(img, cmap="gray") if col_idx < 3 else ax.imshow(img, cmap="magma")
            ax.set_title(title)
            ax.set_xticks([])
            ax.set_yticks([])
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

## 8. Activation Sanity Check

This captures output shapes for leaf modules on one sample. It is useful for spotting exploding activations or unexpected model shapes.

In [ ]:
activation_rows = []
hooks = []

def make_hook(name: str):
    def hook(_module, _inputs, output):
        y = output[0] if isinstance(output, tuple) else output
        if torch.is_tensor(y):
            activation_rows.append((name, tuple(y.shape), float(y.detach().float().mean().cpu()), float(y.detach().float().std(unbiased=False).cpu())))
    return hook

for name, module in model.named_modules():
    if name and not any(module.children()):
        hooks.append(module.register_forward_hook(make_hook(name)))

batch = next(iter(eval_loader))
_ = model(batch["input"].to(device))
for hook in hooks:
    hook.remove()

print(f"captured {len(activation_rows)} leaf activations")
for name, shape, mean, std in activation_rows[:40]:
    print(f"{name:55s} shape={str(shape):22s} mean={mean:+.4g} std={std:.4g}")